In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(r"C:\Users\HP\OneDrive\Desktop\supermarket_analysis\data\superstore.csv")


In [ ]:
df.head(3)

,Category,City,Country,Customer.ID,Customer.Name,Discount,Market,记录数,Order.Date,Order.ID,...,Sales,Segment,Ship.Date,Ship.Mode,Shipping.Cost,State,Sub.Category,Year,Market2,weeknum
0,Office Supplies,Los Angeles,United States,LS-172304,Lycoris Saunders,0.0,US,1,2011-01-07 00:00:00.000,CA-2011-130813,...,19,Consumer,2011-01-09 00:00:00.000,Second Class,4.37,California,Paper,2011,North America,2
1,Office Supplies,Los Angeles,United States,MV-174854,Mark Van Huff,0.0,US,1,2011-01-21 00:00:00.000,CA-2011-148614,...,19,Consumer,2011-01-26 00:00:00.000,Standard Class,0.94,California,Paper,2011,North America,4
2,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,2011-08-05 00:00:00.000,CA-2011-118962,...,21,Consumer,2011-08-09 00:00:00.000,Standard Class,1.81,California,Paper,2011,North America,32


In [ ]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 27 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Category        51290 non-null  object 
 1   City            51290 non-null  object 
 2   Country         51290 non-null  object 
 3   Customer.ID     51290 non-null  object 
 4   Customer.Name   51290 non-null  object 
 5   Discount        51290 non-null  float64
 6   Market          51290 non-null  object 
 7   记录数             51290 non-null  int64  
 8   Order.Date      51290 non-null  object 
 9   Order.ID        51290 non-null  object 
 10  Order.Priority  51290 non-null  object 
 11  Product.ID      51290 non-null  object 
 12  Product.Name    51290 non-null  object 
 13  Profit          51290 non-null  float64
 14  Quantity        51290 non-null  int64  
 15  Region          51290 non-null  object 
 16  Row.ID          51290 non-null  int64  
 17  Sales           51290 non-null 

In [ ]:
drops=['Row.ID','记录数','Market2','weeknum']
df.drop(columns=drops,inplace=True,errors='ignore')

In [ ]:
df.shape

(51290, 23)

In [ ]:
df.columns=(df.columns.str.strip().str.lower().str.replace(".","_").str.replace(" ","_"))

In [ ]:
df['order_date']=pd.to_datetime(df['order_date'])
df['ship_date']=pd.to_datetime(df['ship_date'])

In [ ]:
df.to_csv("../data/superstore_clean.csv",index=False)
print("Cleaned csv saved!!")

Cleaned csv saved!!


In [ ]:
!pip install psycopg2-binary sqlalchemy

In [ ]:
import psycopg2

conn = psycopg2.connect(
    dbname="superstore_db",
    user="postgres",
    password="learnsql321@",
    host="localhost",
    port="5432"
)
cursor=conn.cursor()
print("✅ psycopg2 connection successful!")



✅ psycopg2 connection successful!


In [ ]:

create_table_sql = """
CREATE TABLE IF NOT EXISTS superstore (
    id SERIAL PRIMARY KEY,
    category TEXT,
    city TEXT,
    country TEXT,
    customer_id TEXT,
    customer_name TEXT,
    discount FLOAT,
    market TEXT,
    order_date DATE,
    order_id TEXT,
    order_priority TEXT,
    product_id TEXT,
    product_name TEXT,
    profit FLOAT,
    quantity INTEGER,
    region TEXT,
    sales FLOAT,
    segment TEXT,
    ship_date DATE,
    ship_mode TEXT,
    shipping_cost FLOAT,
    state TEXT,
    sub_category TEXT,
    year INTEGER
);
"""

cursor.execute(create_table_sql)
conn.commit()
print("✅ Table 'superstore' created successfully.")


✅ Table 'superstore' created successfully.


In [ ]:
from tqdm import tqdm

columns = df.columns.tolist()

insert_query = f"""
INSERT INTO superstore ({", ".join(columns)})
VALUES ({", ".join(["%s"] * len(columns))})
"""

for row in tqdm(df.itertuples(index=False, name=None), total=len(df)):
    cursor.execute(insert_query, row)

conn.commit()
print("✅ All rows inserted into PostgreSQL successfully!")


100%|██████████████████████████████████████████████████████████████████████████| 51290/51290 [00:25<00:00, 2017.98it/s]

✅ All rows inserted into PostgreSQL successfully!


In [ ]:
cursor.execute("Select * from superstore limit 5;")
rows=cursor.fetchall()
for row in rows:
    print(row,"\n")

(1, 'Office Supplies', 'Los Angeles', 'United States', 'LS-172304', 'Lycoris Saunders', 0.0, 'US', datetime.date(2011, 1, 7), 'CA-2011-130813', 'High', 'OFF-PA-10002005', 'Xerox 225', 9.3312, 3, 'West', 19.0, 'Consumer', datetime.date(2011, 1, 9), 'Second Class', 4.37, 'California', 'Paper', 2011) 

(2, 'Office Supplies', 'Los Angeles', 'United States', 'MV-174854', 'Mark Van Huff', 0.0, 'US', datetime.date(2011, 1, 21), 'CA-2011-148614', 'Medium', 'OFF-PA-10002893', 'Wirebound Service Call Books, 5 1/2" x 4"', 9.2928, 2, 'West', 19.0, 'Consumer', datetime.date(2011, 1, 26), 'Standard Class', 0.94, 'California', 'Paper', 2011) 

(3, 'Office Supplies', 'Los Angeles', 'United States', 'CS-121304', 'Chad Sievert', 0.0, 'US', datetime.date(2011, 8, 5), 'CA-2011-118962', 'Medium', 'OFF-PA-10000659', 'Adams Phone Message Book, Professional, 400 Message Capacity, 5 3/6” x 11”', 9.8418, 3, 'West', 21.0, 'Consumer', datetime.date(2011, 8, 9), 'Standard Class', 1.81, 'California', 'Paper', 2011)